# Lesson 12.8 - Privacy-Preserving ML/LLM Ops Demo

## Learning Objectives
- Simulate federated learning rounds with local clients.
- Add a simple differential-privacy style noise step.
- Track utility vs privacy trade-off signals.

In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1) Create Decentralized Client Data Splits

In [2]:
X, y = make_classification(n_samples=1200, n_features=12, n_informative=7, random_state=42)
clients = 4
chunks = np.array_split(np.arange(len(X)), clients)
client_sets = [(X[idx], y[idx]) for idx in chunks]
print([c[0].shape for c in client_sets])

[(300, 12), (300, 12), (300, 12), (300, 12)]


## 2) Local Client Training

In [3]:
def train_local_model(X_local, y_local):
    m = LogisticRegression(max_iter=400)
    m.fit(X_local, y_local)
    return m.coef_.copy(), m.intercept_.copy()

local_updates = [train_local_model(xc, yc) for xc, yc in client_sets]
len(local_updates)

4

## 3) Secure-Aggregation Style Averaging (Toy)

In [4]:
coef_avg = np.mean([u[0] for u in local_updates], axis=0)
int_avg = np.mean([u[1] for u in local_updates], axis=0)

global_model = LogisticRegression(max_iter=1)
# dummy fit for shape init
global_model.fit(X[:20], y[:20])
global_model.coef_ = coef_avg
global_model.intercept_ = int_avg

pred = global_model.predict(X)
acc = accuracy_score(y, pred)
print({"global_accuracy": round(acc, 4)})

{'global_accuracy': 0.7775}


/home/ahmad/AI/Github/ai-engineering-zero-to-mastery/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4) Differential-Privacy Style Noise Injection (Toy)

In [5]:
def add_dp_noise(arr, sigma=0.03):
    return arr + np.random.normal(0, sigma, size=arr.shape)

coef_noisy = add_dp_noise(coef_avg, sigma=0.05)
int_noisy = add_dp_noise(int_avg, sigma=0.05)

dp_model = LogisticRegression(max_iter=1)
dp_model.fit(X[:20], y[:20])
dp_model.coef_ = coef_noisy
dp_model.intercept_ = int_noisy

dp_acc = accuracy_score(y, dp_model.predict(X))
print({"accuracy_clean": round(acc, 4), "accuracy_noisy": round(dp_acc, 4)})

{'accuracy_clean': 0.7775, 'accuracy_noisy': 0.7575}


/home/ahmad/AI/Github/ai-engineering-zero-to-mastery/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 5) Governance Logging Stub

In [6]:
audit_log = pd.DataFrame([
    {"round": 1, "clients": clients, "secure_agg": True, "dp_sigma": 0.05, "approved": True}
])
audit_log

,round,clients,secure_agg,dp_sigma,approved
0,1,4,True,0.05,True


## Connect to Theory
- Federated loops avoid central raw-data pooling.
- DP/noise can reduce utility while improving privacy posture.
- Governance requires explicit audit fields and policy thresholds.

## Case Studies & Exceptions
1. Hospital consortium: federated model improved coverage with local data control.
2. Mobile personalization: on-device adaptation plus aggregated updates.
3. Exception: highly non-IID client data can hurt convergence without personalization strategies.

## Interview Questions & Answers
1. **Q:** What is federated learning? **A:** Decentralized training with aggregated updates instead of raw data sharing.
2. **Q:** Why secure aggregation? **A:** To hide individual client updates.
3. **Q:** Differential privacy role? **A:** Limits individual record influence via calibrated noise.
4. **Q:** Main trade-off? **A:** Privacy strength vs model utility/performance.
5. **Q:** Why audit logs? **A:** Compliance and incident traceability.
6. **Q:** Is FL enough for privacy? **A:** No, combine with governance and access controls.
7. **Q:** Horizontal FL? **A:** Same features, different samples across clients.
8. **Q:** Vertical FL? **A:** Same entities, different features by party.
9. **Q:** Deployment guardrail example? **A:** Block promotion if privacy budget threshold is exceeded.
10. **Q:** One-line guidance? **A:** Engineer privacy constraints as measurable system requirements, not afterthoughts.